# GNN-RE: Graph Neural Networks for Gate-Level Netlist Reverse Engineering
### Interactive End-to-End Pipeline, Circuit Graph Visualization & Hardware Security Analysis
**Based on NYUAD Research (IEEE TCAD 2021)**

---

> **💡 Layman Overview:** Think of a flattened chip netlist like an unlabelled map of a secret city where all street signs and building names have been erased. GNN-RE looks at the *traffic patterns* and *interconnections* between logic gates to detect which gates form the **Adder (Calculator +)**, **Multiplier (Calculator ×)**, **Subtractor (-)**, **Comparator (><)**, and **Control Logic (Traffic Lights)**.

> **🎓 Academic / Professor Summary:** Functional reverse engineering formulated as an **Inductive Node Classification** problem on a Directed Acyclic Graph (DAG) $G = (V, E, X)$. Logic gates are nodes, nets are edges, and $X \in \mathbb{R}^{N \times 34}$ captures gate functionality and topological degree signatures. Multi-hop Graph Convolutions recover module boundaries without requiring dynamic simulation.

## 📦 STEP 1: Environment Setup & Core Dependencies
Run the cell below to load all scientific computing and GNN-RE modules.

In [ ]:
# Verify and import dependencies
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import glob, os, json

from netlist_graph_engine import parse_verilog_netlist, build_circuit_graph, CLASS_NAMES, CLASS_COLORS
from gnn_engine import CircuitGNN, extract_subcircuit_boundaries

print("✅ GNN-RE Core Modules loaded successfully!")

## 🔍 STEP 2: Netlist-to-Graph Parsing & Feature Extraction
Let us load a gate-level Verilog benchmark (e.g., `Train_add_mul_combine_4_bit_Syn_65nm.v`) and inspect how raw hardware code converts to a mathematical graph.

In [ ]:
circuit_path = "GNN-RE/Netlist_to_graph/Circuits_datasets/Interconnected-Modules/Train_add_mul_combine_4_bit_Syn_65nm.v"

parsed = parse_verilog_netlist(circuit_path)
nodes, edges, feats, labels = build_circuit_graph(parsed)

print(f"Module Name      : {parsed['module_name']}")
print(f"Total Logic Gates: {len(nodes)}")
print(f"Total Net Wires  : {len(edges)}")
print(f"Feature Matrix   : {feats.shape} (34 features per gate)")
print(f"Target Labels    : {labels.shape}")

print("\n--- Sample Gate Node Extraction ---")
for n in nodes[:5]:
    print(f"Gate ID: {n['id']:2d} | Instance: {n['label']:20s} | Cell Type: {n['cell_type']:15s} | Class: {n['class_name']}")

## 📊 STEP 3: Interactive Circuit Topology Visualization
We render the circuit graph using NetworkX and Matplotlib, coloring each gate according to its functional module.

In [ ]:
G = nx.DiGraph()
for n in nodes:
    G.add_node(n["id"], label=n["label"], color=n["color"], name=n["class_name"])
G.add_edges_from(edges)

plt.figure(figsize=(12, 8), dpi=120)
pos = nx.spring_layout(G, seed=42, k=0.35)

node_colors = [nodes[node]["color"] for node in G.nodes()]
node_sizes = [150 + (nodes[node]["in_degree"] + nodes[node]["out_degree"]) * 40 for node in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, edgecolors="white", linewidths=1.2)
nx.draw_networkx_edges(G, pos, edge_color="#64748b", arrows=True, arrowsize=10, alpha=0.6, width=1.0)

for c_id, name in enumerate(CLASS_NAMES):
    plt.scatter([], [], c=CLASS_COLORS[c_id], label=name, s=100, edgecolors="white")

plt.title(f"Circuit Graph Topology: {parsed['module_name']} ({len(nodes)} Gates)", fontsize=14, fontweight="bold", pad=15)
plt.legend(loc="upper right", frameon=True, facecolor="#f8fafc", edgecolor="#cbd5e1", title="Functional Sub-circuits")
plt.axis("off")
plt.tight_layout()
plt.show()

## 🚀 STEP 4: GNN Model Training & Message Passing
We train our multi-layer GCN across interconnected benchmark netlists.

In [ ]:
model = CircuitGNN(in_dim=34, hidden_dim=64, num_classes=5, depth=2, lr=0.02)

print("--- Training GNN on Interconnected Netlists ---")
train_files = glob.glob("GNN-RE/Netlist_to_graph/Circuits_datasets/Interconnected-Modules/Train_*.v")

history = []
for epoch in range(1, 21):
    epoch_losses = []
    epoch_accs = []
    for fpath in train_files[:10]:
        p = parse_verilog_netlist(fpath)
        if p and len(p["gates"]) > 0:
            _, e, f, l = build_circuit_graph(p)
            loss, acc = model.train_epoch(f, e, l)
            epoch_losses.append(loss)
            epoch_accs.append(acc)
    avg_loss = np.mean(epoch_losses)
    avg_acc = np.mean(epoch_accs)
    history.append((epoch, avg_loss, avg_acc))
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:2d} | Avg Loss: {avg_loss:.4f} | Training Node Accuracy: {avg_acc*100:.2f}%")

epochs, losses, accs = zip(*history)
fig, ax1 = plt.subplots(figsize=(10, 4), dpi=100)
ax1.plot(epochs, losses, color="#ef4444", marker="o", label="Training Loss", linewidth=2)
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Cross-Entropy Loss", color="#ef4444", fontsize=11)
ax1.grid(True, linestyle="--", alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(epochs, [a*100 for a in accs], color="#10b981", marker="s", label="Accuracy (%)", linewidth=2)
ax2.set_ylabel("Accuracy (%)", color="#10b981", fontsize=11)
plt.title("GNN-RE Convergence: Loss and Accuracy vs. Training Epochs", fontsize=13, fontweight="bold")
plt.show()

## 🎯 STEP 5: Testing, Evaluation & Sub-circuit Boundary Extraction
Now we evaluate the trained GNN on an unseen multi-function test netlist.

In [ ]:
test_path = "GNN-RE/Netlist_to_graph/Circuits_datasets/Interconnected-Modules/Test_add_mul_combine_16_bit_Syn_65nm.v"
test_parsed = parse_verilog_netlist(test_path)
test_nodes, test_edges, test_feats, test_labels = build_circuit_graph(test_parsed)

eval_results = model.evaluate(test_feats, test_edges, test_labels)

print(f"=== EVALUATION REPORT ON UNSEEN TEST NETLIST ===")
print(f"Target Design : {test_parsed['module_name']}")
print(f"Gate Count    : {len(test_nodes)}")
print(f"Test Accuracy : {eval_results['accuracy']*100:.2f}%")
print(f"Micro-F1 Score: {eval_results['f1_micro']*100:.2f}%")
print(f"Macro-F1 Score: {eval_results['f1_macro']*100:.2f}%")
print(f"Precision     : {eval_results['precision']*100:.2f}%")
print(f"Recall        : {eval_results['recall']*100:.2f}%")

# Extract Sub-circuit Boundaries
subcircuits = extract_subcircuit_boundaries(test_nodes, test_edges, eval_results["predictions"])
print(f"\nExtracted {len(subcircuits)} Functional Sub-circuits:")
for idx, sc in enumerate(subcircuits[:6]):
    print(f" - Module #{idx+1:2d}: {sc['class_name']:15s} | Size: {sc['size']:4d} gates | Sample Gates: {sc['gates'][:3]}")

## 🌟 STEP 6: Launching the Interactive Web UI (Figma Vision)
To launch the interactive visual web application in your browser, run:
```bash
python web_dashboard.py
```
Then open **http://localhost:8501** in your browser!